# Regression with a smooth equality

Predict the price of a house from six features. Regression in logic is a single formula: "the predicted price **equals** the observed one", with a smooth equality predicate whose truth decreases with the error. Maximising its truth fits the regressor.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import sltn
from sltn import Signature, Interpretation

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data

414 real-estate transactions: transaction date, house age, distance to the nearest MRT station, number of convenience stores, latitude, longitude, and the price per unit area. They are shuffled and split into 330 for training and 84 for testing; the features are standardised with the training mean and deviation, since their scales differ by orders of magnitude.

In [ ]:
data = pd.read_csv("../datasets/real-estate.csv").sample(frac=1, random_state=0)
features = torch.tensor(data.iloc[:, 1:7].to_numpy()).float()
prices = torch.tensor(data[["Y house price of unit area"]].to_numpy()).float()

mean, std = features[:330].mean(0), features[:330].std(0)
features = (features - mean) / std
x_train, y_train = features[:330].to(device), prices[:330].to(device)
x_test, y_test = features[330:].to(device), prices[330:].to(device)
print(data.columns[1:7].tolist())
print(x_train.shape, x_test.shape)

## 1 — Signature

- sorts `Sample` (six features) and `Price`;
- a function `f: Sample → Price`, the regressor;
- a predicate `Eq(u, v)` on prices;
- variables `x` for the samples and `y` for their prices.

In [ ]:
sig = Signature("regression")
sig.sort("Sample")
sig.sort("Price")
sig.function("f", ["Sample"], "Price")
sig.predicate("==", ["Price", "Price"], infix=True)
sig.variable("x", "Sample")
sig.variable("y", "Price")
print(sig)

## 2 — Knowledge base

```
∀ (x, y): Eq(f(x), y)
```

The quantifier is **diagonal** (Tutorial 2 §3): `x` and `y` hold the features and prices of the same houses, row by row, so each prediction is compared with its own price, not with all of them.

In [ ]:
phi = sig.parse("∀ (x, y): y==f(x)")
phi

## 3 — Model

An MLP with a linear output, since a price is not a truth value.

In [ ]:
class Regressor(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(6, 8), torch.nn.ELU(),
            torch.nn.Linear(8, 8), torch.nn.ELU(),
            torch.nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.net(x)

## 4 — Interpretation

`Eq(u, v) = exp(-α |u - v|)`, with α = 0.05: 1 for a perfect prediction, about 0.6 for an error of 10 units. α sets how strict the equality is; prices here range from about 10 to 80.

The ∀ aggregator matters too. With the default pMeanError (p = 2), the loss is the mean of $(1 - \mathrm{Eq})^2$, so large errors count more, as in a squared loss.

In [ ]:
alpha = 0.05
interp = Interpretation(sig).to(device)
interp["Sample"] = sltn.Type("Sample", shape=(6,))
interp["Price"] = sltn.Type("Price", shape=(1,))
interp["f"] = Regressor()
interp["=="] = lambda u, v: torch.exp(-alpha * torch.abs(u - v).sum(-1, keepdim=True))
print(interp)

## 5 — Metrics

Truth of the formula, and the root-mean-square error of the predictions.

In [ ]:
def evaluate(x, y):
    interp.eval()
    with torch.no_grad():
        interp["x"], interp["y"] = x, y
        sat = interp(phi).value.item()
        rmse = (interp["f"](x) - y).pow(2).mean().sqrt().item()
    return sat, rmse

## 6 — Training

The loss is $1 - \text{truth}(\phi)$, on minibatches of 64 houses.

In [ ]:
optimizer = torch.optim.Adam(interp.parameters(), lr=0.005)

for epoch in range(500):
    interp.train()
    for idx in torch.randperm(len(x_train)).split(64):
        interp["x"], interp["y"] = x_train[idx], y_train[idx]
        optimizer.zero_grad()
        loss = 1 - interp(phi).value
        loss.backward()
        optimizer.step()

    if epoch % 100 == 0 or epoch == 499:
        sat_train, rmse_train = evaluate(x_train, y_train)
        sat_test, rmse_test = evaluate(x_test, y_test)
        print(f"epoch {epoch:3d} | sat train {sat_train:.3f} test {sat_test:.3f} | RMSE train {rmse_train:.2f} test {rmse_test:.2f}")

## 7 — Results

Predicted against observed prices on the test set.

In [ ]:
interp.eval()
with torch.no_grad():
    predicted = interp["f"](x_test).cpu()

plt.figure(figsize=(4.5, 4.5))
plt.scatter(y_test.cpu(), predicted, s=12)
lim = [0, max(y_test.max().item(), predicted.max().item()) + 5]
plt.plot(lim, lim, "k--", lw=1)
plt.xlabel("observed price"); plt.ylabel("predicted price")
plt.title("Test set")
plt.show()

The test RMSE settles well below the standard deviation of the prices. The points follow the diagonal; the largest errors are a few very expensive houses, which the model underestimates.

## Summary

- Regression is the formula `∀ (x, y): Eq(f(x), y)` with a smooth equality; maximising its truth fits `f`.
- The diagonal quantifier pairs each sample with its own target.
- The strictness of `Eq` (α) and the ∀ aggregator play the role of the loss function.
- A function symbol can return any value (here a price), not only a truth value.